## 1- Import & load Raw Data

In [ ]:
import pandas as pd
import numpy as np

Source_File = '../Data/Central_Superstore.xlsx'
Sheet_Name = 'Central_Region'

raw_data = pd.read_excel(Source_File, sheet_name = Sheet_Name)

print(f"Raw Data's Shape: {raw_data.shape}")
print( raw_data.dtypes)
raw_data.head()

Raw Data's Shape: (2323, 21)
Row ID                    int64
Order ID                 object
Order Date       datetime64[ns]
Ship Date        datetime64[ns]
Ship Mode                object
Customer ID              object
Customer Name            object
Segment                  object
Country                  object
City                     object
State                    object
Postal Code               int64
Region                   object
Product ID               object
Category                 object
Sub-Category             object
Product Name             object
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
dtype: object


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,15,US-2012-118983,2014-11-22,2014-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,...,76106,Central,OFF-AP-10002311,Office Supplies,Appliances,Holmes Replacement Filter for HEPA Air Cleaner...,68.810,5,0.8,-123.8580
1,16,US-2012-118983,2014-11-22,2014-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,...,76106,Central,OFF-BI-10000756,Office Supplies,Binders,Storex DuraTech Recycled Plastic Frosted Binders,2.544,3,0.8,-3.8160
2,17,CA-2011-105893,2013-11-11,2013-11-18,Standard Class,PK-19075,Pete Kriz,Consumer,United States,Madison,...,53711,Central,OFF-ST-10004186,Office Supplies,Storage,"Stur-D-Stor Shelving, Vertical 5-Shelf: 72""H x...",665.880,6,0.0,13.3176
3,22,CA-2013-137330,2015-12-10,2015-12-14,Standard Class,KB-16585,Ken Black,Corporate,United States,Fremont,...,68025,Central,OFF-AR-10000246,Office Supplies,Art,Newell 318,19.460,7,0.0,5.0596
4,23,CA-2013-137330,2015-12-10,2015-12-14,Standard Class,KB-16585,Ken Black,Corporate,United States,Fremont,...,68025,Central,OFF-AP-10001492,Office Supplies,Appliances,"Acco Six-Outlet Power Strip, 4' Cord Length",60.340,7,0.0,15.6884


## 2- Profiling Data before Cleaning

In [51]:
# Check Nulls
raw_data.isnull().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [52]:
# check Duplicates
raw_data.duplicated().sum()

np.int64(0)

In [53]:
# Check Uniqueness
raw_data.nunique()

Row ID           2323
Order ID         1175
Order Date        720
Ship Date         746
Ship Mode           4
Customer ID       629
Customer Name     629
Segment             3
Country             1
City              181
State              13
Postal Code       195
Region              1
Product ID       1310
Category            3
Sub-Category       17
Product Name     1295
Sales            1955
Quantity           14
Discount            9
Profit           2088
dtype: int64

### Primary Key Validation for each Dimension

In [54]:
# Customer Profiling
cus_pro = raw_data.groupby('Customer ID')[['Customer Name','Segment']].nunique()
print("Customer IDs with Name > 1:",(cus_pro['Customer Name'] > 1).sum())
print("Customer IDs with Segment > 1:",(cus_pro['Segment'] > 1).sum())

Customer IDs with Name > 1: 0
Customer IDs with Segment > 1: 0


In [55]:
# Product Profiling
product_pro = raw_data.groupby('Product ID')[['Product Name','Category','Sub-Category']].nunique()
print("Product IDs with Name > 1:",(product_pro['Product Name'] > 1).sum())
print("Product IDs with Category > 1:",(product_pro['Category'] > 1).sum())
print("Product IDs with Sub-Category > 1:",(product_pro['Sub-Category'] > 1).sum())

Product IDs with Name > 1: 16
Product IDs with Category > 1: 0
Product IDs with Sub-Category > 1: 0


In [56]:
order_level = raw_data.groupby('Order ID')[['Order Date','Ship Date','Ship Mode','Customer ID']].nunique()
(order_level > 1).sum()

Order Date     0
Ship Date      0
Ship Mode      0
Customer ID    0
dtype: int64

## 3- Data Cleaning

In [57]:
raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2323 entries, 0 to 2322
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         2323 non-null   int64         
 1   Order ID       2323 non-null   object        
 2   Order Date     2323 non-null   datetime64[ns]
 3   Ship Date      2323 non-null   datetime64[ns]
 4   Ship Mode      2323 non-null   object        
 5   Customer ID    2323 non-null   object        
 6   Customer Name  2323 non-null   object        
 7   Segment        2323 non-null   object        
 8   Country        2323 non-null   object        
 9   City           2323 non-null   object        
 10  State          2323 non-null   object        
 11  Postal Code    2323 non-null   int64         
 12  Region         2323 non-null   object        
 13  Product ID     2323 non-null   object        
 14  Category       2323 non-null   object        
 15  Sub-Category   2323 n

In [58]:
# Get Copy to make Edits
df = raw_data.copy()

In [59]:
# Change Columns's names to snake_case
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ","_", regex=False)
    .str.replace("-","_", regex=False)
    .str.lower()
)

In [60]:
df.columns

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'sub_category',
       'product_name', 'sales', 'quantity', 'discount', 'profit'],
      dtype='object')

In [ ]:
# Trim white spaces for all textual columns
text_cols = ['order_id', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'region', 'product_id', 'category', 'sub_category',
       'product_name']
for c in text_cols:
    df[c] = df[c].astype(str).str.strip()

In [62]:
# Check Numeric Columns
df[df['sales']<=0]

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit


In [63]:
df[df['quantity']<=0]

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit


In [64]:
df['discount'].unique()

array([0.8 , 0.  , 0.2 , 0.6 , 0.32, 0.3 , 0.5 , 0.4 , 0.1 ])

In [65]:
df['profit'].max()

8399.975999999999

In [66]:
df['profit'].min()

-3701.8928000000014

## 4- Building Dimensions Tables

### 4-1 Date Dimension

In [67]:
dates = pd.concat([df['order_date'],df['ship_date']]).drop_duplicates().sort_values()

In [68]:
dates.head()

1809   2013-01-03
188    2013-01-04
1752   2013-01-07
188    2013-01-08
152    2013-01-09
dtype: datetime64[ns]

In [69]:
dim_date = pd.DataFrame({"full_date":dates})

In [70]:
dim_date

,full_date
1809,2013-01-03
188,2013-01-04
1752,2013-01-07
188,2013-01-08
152,2013-01-09
...,...
163,2016-12-30
237,2017-01-01
1716,2017-01-02
991,2017-01-04


In [71]:
dim_date['date_key'] = dim_date['full_date'].dt.strftime("%Y%m%d").astype(int)
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['quarter'] = dim_date['full_date'].dt.quarter
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['month_name'] = dim_date['full_date'].dt.month_name()
dim_date['day'] = dim_date['full_date'].dt.day
dim_date['day_name'] = dim_date['full_date'].dt.day_name()
dim_date['week_of_year'] = dim_date['full_date'].dt.isocalendar().week.astype(int)
dim_date['is_weekend'] = dim_date['full_date'].dt.dayofweek >= 5

In [72]:
dim_date['is_weekend'] = dim_date['is_weekend'].astype(str).replace({"False":"No","True":"Yes"})

In [73]:
dim_date = dim_date[['date_key','full_date','year','quarter','month','month_name','day','day_name','week_of_year','is_weekend']].reset_index(drop=True)

In [74]:
dim_date.head(3)

,date_key,full_date,year,quarter,month,month_name,day,day_name,week_of_year,is_weekend
0,20130103,2013-01-03,2013,1,1,January,3,Thursday,1,No
1,20130104,2013-01-04,2013,1,1,January,4,Friday,1,No
2,20130107,2013-01-07,2013,1,1,January,7,Monday,2,No


### 4-2 Customer Dimension

In [75]:
dim_customer = df[['customer_id','customer_name','segment']].drop_duplicates().sort_values('customer_id').reset_index(drop=True)

In [76]:
dim_customer.insert(0,"customer_key",dim_customer.index + 1)

In [77]:
dim_customer.head(3)

,customer_key,customer_id,customer_name,segment
0,1,AA-10315,Alex Avila,Consumer
1,2,AA-10375,Allen Armold,Consumer
2,3,AA-10480,Andrew Allen,Consumer


In [78]:
dim_customer.shape

(629, 4)

### 4-3 Product Dimension

In [79]:
dim_product = (
    df[['product_id','product_name','category','sub_category']].drop_duplicates()
    .sort_values(['product_id','product_name']).reset_index(drop=True)
)
dim_product.insert(0,"product_key",dim_product.index + 1)

In [80]:
dim_product.head(3)

,product_key,product_id,product_name,category,sub_category
0,1,FUR-BO-10000112,"Bush Birmingham Collection Bookcase, Dark Cherry",Furniture,Bookcases
1,2,FUR-BO-10000362,Sauder Inglewood Library Bookcases,Furniture,Bookcases
2,3,FUR-BO-10000468,O'Sullivan 2-Shelf Heavy-Duty Bookcases,Furniture,Bookcases


### 4-4 Location Dimention

In [81]:
df.columns

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'sub_category',
       'product_name', 'sales', 'quantity', 'discount', 'profit'],
      dtype='object')

In [82]:
dim_location = (
    df[['country', 'city', 'state', 'region', 'postal_code']].drop_duplicates().sort_values(['country', 'city', 'state'
      ,'region' ,'postal_code']).reset_index(drop=True)
)

In [83]:
dim_location.insert(0,"location_key",dim_location.index +1)

In [84]:
dim_location

,location_key,country,city,state,region,postal_code
0,1,United States,Aberdeen,South Dakota,Central,57401
1,2,United States,Abilene,Texas,Central,79605
2,3,United States,Allen,Texas,Central,75002
3,4,United States,Amarillo,Texas,Central,79109
4,5,United States,Ann Arbor,Michigan,Central,48104
...,...,...,...,...,...,...
190,191,United States,Westland,Michigan,Central,48185
191,192,United States,Wheeling,Illinois,Central,60090
192,193,United States,Wichita,Kansas,Central,67212
193,194,United States,Woodbury,Minnesota,Central,55125


### 4-5 Ship Mode Dimension

In [85]:
dim_ship_mode = df[['ship_mode']].drop_duplicates().sort_values('ship_mode').reset_index(drop=True)

dim_ship_mode.insert(0,"ship_mode_key",dim_ship_mode.index +1)

In [86]:
dim_ship_mode

,ship_mode_key,ship_mode
0,1,First Class
1,2,Same Day
2,3,Second Class
3,4,Standard Class


## 5 Building Fact Table

In [87]:
df.head(1)

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,15,US-2012-118983,2014-11-22,2014-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,...,76106,Central,OFF-AP-10002311,Office Supplies,Appliances,Holmes Replacement Filter for HEPA Air Cleaner...,68.81,5,0.8,-123.858


In [88]:
dim_date.head(1)

,date_key,full_date,year,quarter,month,month_name,day,day_name,week_of_year,is_weekend
0,20130103,2013-01-03,2013,1,1,January,3,Thursday,1,No


In [89]:
fact_table = df.merge(dim_customer, on = ["customer_id","customer_name","segment"], how = "left") \
                .merge(dim_product, on = ["product_id","category","sub_category","product_name"], how = "left") \
                .merge(dim_location, on = ["country", "city", "state","postal_code", "region"], how = "left") \
                .merge(dim_ship_mode, on = ["ship_mode"], how = "left") \
                .merge(dim_date[["date_key","full_date"]].rename(columns={"date_key":"order_date_key","full_date":"order_date"}),
                 on = ["order_date"], how = "left") \
                .merge(dim_date[["date_key","full_date"]].rename(columns={"date_key":"ship_date_key","full_date":"ship_date"}),
                 on = ["ship_date"], how = "left")

In [90]:
fact_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2323 entries, 0 to 2322
Data columns (total 27 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   row_id          2323 non-null   int64         
 1   order_id        2323 non-null   object        
 2   order_date      2323 non-null   datetime64[ns]
 3   ship_date       2323 non-null   datetime64[ns]
 4   ship_mode       2323 non-null   object        
 5   customer_id     2323 non-null   object        
 6   customer_name   2323 non-null   object        
 7   segment         2323 non-null   object        
 8   country         2323 non-null   object        
 9   city            2323 non-null   object        
 10  state           2323 non-null   object        
 11  postal_code     2323 non-null   int64         
 12  region          2323 non-null   object        
 13  product_id      2323 non-null   object        
 14  category        2323 non-null   object        
 15  sub_

In [91]:
fact_table.columns

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'sub_category',
       'product_name', 'sales', 'quantity', 'discount', 'profit',
       'customer_key', 'product_key', 'location_key', 'ship_mode_key',
       'order_date_key', 'ship_date_key'],
      dtype='object')

In [92]:
fact_table = fact_table[['row_id', 'order_id', 'order_date_key', 'ship_date_key','ship_mode_key',  'customer_key', 'product_key',
                         'location_key','sales', 'quantity', 'discount', 'profit',]] \
                        .rename(columns={"row_id":"sales_id"}) \
                        .sort_values("sales_id").reset_index(drop=True)

In [93]:
fact_table.head()

,sales_id,order_id,order_date_key,ship_date_key,ship_mode_key,customer_key,product_key,location_key,sales,quantity,discount,profit
0,15,US-2012-118983,20141122,20141126,4,263,310,63,68.810,5,0.8,-123.8580
1,16,US-2012-118983,20141122,20141126,4,263,479,63,2.544,3,0.8,-3.8160
2,17,CA-2011-105893,20131111,20131118,4,463,1032,109,665.880,6,0.0,13.3176
3,22,CA-2013-137330,20151210,20151214,4,323,354,67,19.460,7,0.0,5.0596
4,23,CA-2013-137330,20151210,20151214,4,323,298,67,60.340,7,0.0,15.6884


In [94]:
fact_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2323 entries, 0 to 2322
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sales_id        2323 non-null   int64  
 1   order_id        2323 non-null   object 
 2   order_date_key  2323 non-null   int64  
 3   ship_date_key   2323 non-null   int64  
 4   ship_mode_key   2323 non-null   int64  
 5   customer_key    2323 non-null   int64  
 6   product_key     2323 non-null   int64  
 7   location_key    2323 non-null   int64  
 8   sales           2323 non-null   float64
 9   quantity        2323 non-null   int64  
 10  discount        2323 non-null   float64
 11  profit          2323 non-null   float64
dtypes: float64(3), int64(8), object(1)
memory usage: 217.9+ KB


## 6- Connecting SQL Server

In [95]:
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings("ignore", category=Warning)

SQL_SERVER = "ABDELAZIM-LAB\MYSQLSERVER2026"
SQL_DATABASE = "SuperstoreDW"
SQL_DRIVER = "ODBC Driver 17 for SQL Server"

conn_str = (
    f"mssql+pyodbc://@{SQL_SERVER}/{SQL_DATABASE}"
    f"?driver={SQL_DRIVER.replace(' ', '+')}&trusted_connection=yes"
)

engine = create_engine(conn_str, fast_executemany=True)

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT @@VERSION"))
        print(result.fetchone()[0])
except Exception as e:
    print("Connection Failed", e)

Microsoft SQL Server 2025 (RTM-GDR) (KB5122770) - 17.0.1135.8 (X64) 
	Aug 20 2026 00:56:11 
	Copyright (C) 2025 Microsoft Corporation
	Standard Developer Edition (64-bit) on Windows 10 Pro 10.0 <X64> (Build 22631: )



In [96]:
fact_table.columns

Index(['sales_id', 'order_id', 'order_date_key', 'ship_date_key',
       'ship_mode_key', 'customer_key', 'product_key', 'location_key', 'sales',
       'quantity', 'discount', 'profit'],
      dtype='object')

## 7- Create Schema & Tables

In [102]:
ddl_statements = [
     """
     CREATE TABLE dbo.dim_customer(
     customer_key INT PRIMARY KEY,
     customer_id VARCHAR(20) NOT NULL,
     customer_name VARCHAR(50) NOT NULL,
     segment VARCHAR(20) NOT NULL
     );
     """
    ,
     """
     CREATE TABLE dbo.dim_product(
     product_key INT PRIMARY KEY,
     product_id VARCHAR(20) NOT NULL,
     product_name VARCHAR(200) NOT NULL,
     category VARCHAR(200) NOT NULL,
     sub_category VARCHAR(200) NOT NULL
     );
     """
    ,
     """
     CREATE TABLE dbo.dim_date(
     date_key INT PRIMARY KEY,
     full_date DATE NOT NULL,
     year SMALLINT NOT NULL,
     quarter TINYINT NOT NULL,
     month SMALLINT NOT NULL,
     month_name VARCHAR(15) NOT NULL,
     day SMALLINT NOT NULL,
     day_name VARCHAR(15) NOT NULL,
     week_of_year SMALLINT NOT NULL,
     is_weekend VARCHAR(5) NOT NULL
     );
     """
    ,
     """
     CREATE TABLE dbo.dim_location(
     location_key INT PRIMARY KEY,
     country VARCHAR(50) NOT NULL,
     city VARCHAR(50) NOT NULL,
     state VARCHAR(50) NOT NULL,
     region VARCHAR(50) NOT NULL,
     postal_code INT NOT NULL
     );
     """
    ,
     """
     CREATE TABLE dbo.dim_ship_mode(
     ship_mode_key INT PRIMARY KEY,
     ship_mode VARCHAR(30) NOT NULL
     );
     """
    ,
     """
     CREATE TABLE dbo.fact_table(
     sales_id  INT PRIMARY KEY,
     order_id VARCHAR(20) NOT NULL,
     order_date_key INT NOT NULL,
     ship_date_key INT NOT NULL,
     ship_mode_key INT NOT NULL,
     customer_key INT NOT NULL,
     product_key INT NOT NULL, 
     location_key INT NOT NULL,
     sales DECIMAL(12,4),
     quantity INT,
     discount DECIMAL(5,4),
     profit DECIMAL(12,4),
     CONSTRAINT fk_fact_order_date FOREIGN KEY (order_date_key) REFERENCES dbo.dim_date(date_key),
     CONSTRAINT fk_fact_ship_date FOREIGN KEY (ship_date_key) REFERENCES dbo.dim_date(date_key),
     CONSTRAINT fk_fact_ship_mode FOREIGN KEY (ship_mode_key) REFERENCES dbo.dim_ship_mode(ship_mode_key),
     CONSTRAINT fk_fact_customer FOREIGN KEY (customer_key) REFERENCES dbo.dim_customer(customer_key),
     CONSTRAINT fk_fact_product FOREIGN KEY (product_key) REFERENCES dbo.dim_product(product_key),
     CONSTRAINT fk_fact_location FOREIGN KEY (location_key) REFERENCES dbo.dim_location(location_key)
     );
     """
]
with engine.begin() as conn:
    for stmt in ddl_statements:
        conn.execute(text(stmt))
print("All Tables are created")

All Tables are created


## 8- Load Data into SQL Server Tables

In [103]:
# DataFrame.to_sql(name, con, *, schema=None, if_exists='fail', chunksize=None)

In [104]:
load_order = [
    ("dim_customer",dim_customer),
    ("dim_product",dim_product),
    ("dim_date",dim_date),
    ("dim_location",dim_location),
    ("dim_ship_mode",dim_ship_mode),
    ("fact_table",fact_table)
]
for table, df in load_order:
    df.to_sql(name = table, con = engine, schema = "dbo", if_exists = "append",
              index = False, chunksize = 1000)
    print(f"loaded {len(df)} rows in dbo.{table}")

loaded 629 rows in dbo.dim_customer
loaded 1326 rows in dbo.dim_product
loaded 1058 rows in dbo.dim_date
loaded 195 rows in dbo.dim_location
loaded 4 rows in dbo.dim_ship_mode
loaded 2323 rows in dbo.fact_table


## 9- Verify loaded Data

In [108]:
query = """
        SELECT *
        FROM dbo.fact_table f
        JOIN dbo.dim_customer c
        ON f.customer_key = c.customer_key
        JOIN dbo.dim_product p
        ON f.product_key = p.product_key
        JOIN dbo.dim_date d1
        ON f.order_date_key = d1.date_key 
        JOIN dbo.dim_date d2
        ON f.ship_date_key = d2.date_key
        JOIN dbo.dim_location l
        ON f.location_key = l.location_key
        jOIN dbo.dim_ship_mode sm
        ON f.ship_mode_key = sm.ship_mode_key
        """

new_df = pd.read_sql(query,engine)
new_df.head(3)

,sales_id,order_id,order_date_key,ship_date_key,ship_mode_key,customer_key,product_key,location_key,sales,quantity,...,week_of_year,is_weekend,location_key,country,city,state,region,postal_code,ship_mode_key,ship_mode
0,15,US-2012-118983,20141122,20141126,4,263,310,63,68.810,5,...,48,No,63,United States,Fort Worth,Texas,Central,76106,4,Standard Class
1,16,US-2012-118983,20141122,20141126,4,263,479,63,2.544,3,...,48,No,63,United States,Fort Worth,Texas,Central,76106,4,Standard Class
2,17,CA-2011-105893,20131111,20131118,4,463,1032,109,665.880,6,...,47,No,109,United States,Madison,Wisconsin,Central,53711,4,Standard Class


In [109]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2323 entries, 0 to 2322
Data columns (total 49 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sales_id        2323 non-null   int64  
 1   order_id        2323 non-null   object 
 2   order_date_key  2323 non-null   int64  
 3   ship_date_key   2323 non-null   int64  
 4   ship_mode_key   2323 non-null   int64  
 5   customer_key    2323 non-null   int64  
 6   product_key     2323 non-null   int64  
 7   location_key    2323 non-null   int64  
 8   sales           2323 non-null   float64
 9   quantity        2323 non-null   int64  
 10  discount        2323 non-null   float64
 11  profit          2323 non-null   float64
 12  customer_key    2323 non-null   int64  
 13  customer_id     2323 non-null   object 
 14  customer_name   2323 non-null   object 
 15  segment         2323 non-null   object 
 16  product_key     2323 non-null   int64  
 17  product_id      2323 non-null   o